# StreamFlix - Phase 1: Data Cleaning

This notebook loads the six project tables, checks data quality, applies simple cleaning, and saves clean CSV files.

In [ ]:
import os
import pandas as pd

REF_DATE = pd.Timestamp("2026-06-01")

subscribers = pd.read_csv("subscribers.csv")
titles = pd.read_csv("titles.csv")
watch_history = pd.read_csv("watch_history.csv")
watchlist = pd.read_csv("watchlist.csv")
ratings = pd.read_csv("ratings.csv")
reviews = pd.read_csv("reviews.csv")

print("Files loaded successfully")


Files loaded successfully


## 1. Basic profile

In [ ]:
tables = {
    "subscribers": subscribers,
    "titles": titles,
    "watch_history": watch_history,
    "watchlist": watchlist,
    "ratings": ratings,
    "reviews": reviews
}

for name, table in tables.items():
    print(name, "-", len(table), "rows and", table.shape[1], "columns")


subscribers - 15000 rows and 14 columns
titles - 9000 rows and 21 columns
watch_history - 650000 rows and 10 columns
watchlist - 65000 rows and 5 columns
ratings - 130000 rows and 5 columns
reviews - 110000 rows and 7 columns


## 2. Convert date columns

In [ ]:
date_columns = {
    "subscribers": ["signup_date", "churn_date"],
    "titles": ["date_added", "license_expiry"],
    "watch_history": ["watch_date"],
    "watchlist": ["added_date"],
    "ratings": ["rating_date"],
    "reviews": ["review_date"]
}

for table_name, columns in date_columns.items():
    for column in columns:
        tables[table_name][column] = pd.to_datetime(
            tables[table_name][column], errors="coerce"
        )

print("Date columns converted")


Date columns converted


## 3. Missing values

In [ ]:
for name, table in tables.items():
    missing = table.isna().sum()
    missing = missing[missing > 0]
    print("\n", name)
    print(missing if len(missing) else "No missing values")



 subscribers
churn_date    11199
dtype: int64

 titles
license_expiry    1966
dtype: int64

 watch_history
No missing values

 watchlist
No missing values

 ratings
No missing values

 reviews
No missing values


## 4. Duplicate checks

In [ ]:
key_columns = {
    "subscribers": "subscriber_id",
    "titles": "title_id",
    "watch_history": "watch_id",
    "watchlist": "watchlist_id",
    "ratings": "rating_id",
    "reviews": "review_id"
}

for name, key in key_columns.items():
    table = tables[name]
    print(name, "duplicate IDs:", table[key].duplicated().sum())


subscribers duplicate IDs: 0
titles duplicate IDs: 0
watch_history duplicate IDs: 0
watchlist duplicate IDs: 0
ratings duplicate IDs: 0
reviews duplicate IDs: 0


## 5. Business-rule checks

In [ ]:
print("Churn date before signup:", (
    subscribers["churn_date"].notna()
    & (subscribers["churn_date"] < subscribers["signup_date"])
).sum())
print("Active subscribers with a churn date:", (
    subscribers["is_active"] & subscribers["churn_date"].notna()
).sum())
print("Churned subscribers without a churn date:", (
    (~subscribers["is_active"]) & subscribers["churn_date"].isna()
).sum())
print("Invalid ratings:", (~ratings["rating"].between(1, 5)).sum())
print("Invalid sentiments:", (~reviews["sentiment"].isin(["Positive", "Neutral", "Negative"])).sum())
print("Sessions longer than content:", (
    watch_history["watch_duration_min"] > watch_history["content_duration_min"]
).sum())


Churn date before signup: 0
Active subscribers with a churn date: 0
Churned subscribers without a churn date: 0
Invalid ratings: 0
Invalid sentiments: 0
Sessions longer than content: 0


## 6. Referential integrity

In [ ]:
subscriber_ids = set(subscribers["subscriber_id"])
title_ids = set(titles["title_id"])

for name in ["watch_history", "watchlist", "ratings", "reviews"]:
    table = tables[name]
    missing_subscribers = (~table["subscriber_id"].isin(subscriber_ids)).sum()
    missing_titles = (~table["title_id"].isin(title_ids)).sum()
    print(name, "missing subscribers:", missing_subscribers, "missing titles:", missing_titles)


watch_history missing subscribers: 0 missing titles: 0
watchlist missing subscribers: 0 missing titles: 0
ratings missing subscribers: 0 missing titles: 0
reviews missing subscribers: 0 missing titles: 0


## 7. Create clean files

In [ ]:
clean = "clean"
os.makedirs(clean, exist_ok=True)

subscribers["churn_after_asof"] = (
    subscribers["churn_date"].notna()
    & (subscribers["churn_date"] > REF_DATE)
)
subscribers["signup_month"] = subscribers["signup_date"].dt.to_period("M").astype(str)

watch_history["completion_check"] = (
    watch_history["watch_duration_min"]
    / watch_history["content_duration_min"]
    * 100
)
watch_history = watch_history.merge(
    subscribers[["subscriber_id", "signup_date", "churn_date"]],
    on="subscriber_id",
    how="left"
)
watch_history["before_signup"] = watch_history["watch_date"] < watch_history["signup_date"]
watch_history["after_churn"] = (
    watch_history["churn_date"].notna()
    & (watch_history["watch_date"] > watch_history["churn_date"])
)
watch_history["long_session_24h"] = (
    watch_history["watch_duration_min"] > 24 * 60
)

watchlist = watchlist.merge(
    subscribers[["subscriber_id", "signup_date", "churn_date"]],
    on="subscriber_id",
    how="left"
)
watchlist["before_signup"] = watchlist["added_date"] < watchlist["signup_date"]
watchlist["after_churn"] = (
    watchlist["churn_date"].notna()
    & (watchlist["added_date"] > watchlist["churn_date"])
)

for name, table in tables.items():
    table.to_csv(f"{clean}/{name}_clean.csv", index=False)

print("Clean files saved in the clean folder")


Clean files saved in the clean folder


## 8. Summary

In [ ]:
print("Subscribers:", len(subscribers))
print("Titles:", len(titles))
print("Watch sessions:", len(watch_history))
print("Active rate:", round(subscribers["is_active"].mean() * 100, 1), "%")
print("Average completion:", round(watch_history["completion_pct"].mean(), 1), "%")
print("Phase 1 complete")


Subscribers: 15000
Titles: 9000
Watch sessions: 650000
Active rate: 74.7 %
Average completion: 65.3 %
Phase 1 complete


## Phase 1 Summary

- The six source tables were loaded successfully, covering 15,000 subscribers, 9,000 titles, 650,000 watch sessions, 65,000 watchlist records, 130,000 ratings, and 110,000 reviews.
- Date fields were converted to datetime values, and the only expected missing values were `churn_date` for active subscribers and `license_expiry` for titles without an expiry value.
- Duplicate-ID checks, business-rule checks, and referential-integrity checks all passed with zero invalid records or broken relationships.
- Clean versions of the datasets were written to the `clean` folder. The resulting baseline shows a 74.7% active-subscriber rate and 65.3% average completion rate, providing a validated foundation for the EDA and KPI phases.